In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from functools import partial
from optimized_svm.newton import solve_twostep

import numpy as np
from scipy.linalg import cho_factor, cho_solve

In [ ]:
def diff(x1, x2, idx):
    x1sqr = (x1[:, idx] * x1[:, idx]).sum(axis=1)
    x2sqr = (x2[:, idx] * x2[:, idx]).sum(axis=1)
    d = x1sqr[:, None] + x2sqr[None, :] - 2.0 * x1[:, idx].dot(x2[:, idx].T)
    return d

def kernel(x1, x2, gamma):
    return np.exp(-gamma * diff(x1, x2, slice(None)))

def dkernel(x1, x2, gamma):
    d = diff(x1, x2, slice(None))
    return -np.exp(-gamma * d) * d

def smooth_max_p2(t, s):
    return np.piecewise(
        t, [t <= -s, t >= s],
        [0.0, lambda t: t, lambda t: 0.25 / s * (t + s)**2]
    )


def dsmooth_max_p2(t, s):
    return np.piecewise(
        t, [t <= -s, t >= s], [0.0, 1.0, lambda t: 0.5 / s * (t + s)]
    )


def ddsmooth_max_p2(t, s):
    return np.piecewise(t, [t <= -s, t >= s], [0.0, 0.0, 0.5 / s])


def log_max(t):
    return np.log(1.0 + np.exp(t))


def dlog_max(t):
    return 1.0 - 1.0 / (1.0 + np.exp(t))


def ddlog_max(t):
    return np.exp(t) / (1.0 + np.exp(t)) ** 2

In [ ]:
np.random.seed(42)

n, nft = 1000, 10
x = np.random.rand(n, nft)
yr = np.sin(np.pi * x).sum(axis=1)
y = np.where(yr > np.median(yr), 1.0, -1.0)

log_lmbda = np.log(0.01)
log_gamma = 0.5 * np.log(1.5)
# smoothingParam = np.log(2.0) / 0.25
# shift = 0.0
smoothingParam = 0.5
shift = 1.0

In [ ]:
maxfun = [
    partial(fun, s=smoothingParam)
    for fun in [smooth_max_p2, dsmooth_max_p2, ddsmooth_max_p2]
]
# maxfun = log_max, dlog_max, ddlog_max

In [ ]:

%%time

z0 = solve_twostep(
    np.exp(log_gamma) * x, y,
    maxfun,
    np.exp(log_lmbda),
    1.0,
    smoothingParam, shift,
)
a0, b0, c0 = z0

log_gamma_n = log_gamma * np.ones(x.shape[1])
# log_gamma_n[0] = log_gamma + 1e-6
gamma_n = np.exp(log_gamma_n)
z1 = solve_twostep(
    gamma_n[None, :] * x, y,
    maxfun,
    np.exp(log_lmbda + 1e-6),
    1.0,
    smoothingParam, shift,
)
a1, b1, c1 = z1

In [ ]:
dan = (a1 - a0) / 1e-6
dbn = (b1 - b0) / 1e-6
dcn = (c1 - c0) / 1e-6

In [ ]:
def solve_rendered(k_p, h_p, rhs_p, lmbda):
    qm = k_p / lmbda + np.diag(1.0 / h_p)

    qm_cho = cho_factor(qm, overwrite_a=True)
    p_p = cho_solve(qm_cho, rhs_p)
    q_p = cho_solve(qm_cho, np.ones_like(rhs_p))

    u_0 = p_p.sum() / q_p.sum()
    u_p = p_p - u_0 * q_p
    return u_p, u_0

In [ ]:
gamma = np.exp(log_gamma)
lmbda = np.exp(log_lmbda)

a, b, c = z0
idx_sv = a != 0

xs = gamma * x
xs_sv = xs[idx_sv]
k = kernel(xs, xs_sv, 1.0)
ka = k.dot(a[idx_sv]) / lmbda

dec = ka + b
f = shift - y * dec
h = maxfun[2](f)

idx_p = h > 0
h_p = h[idx_p]
k_p = kernel(xs[idx_p], xs[idx_p], 1.0)

d = diff(xs[idx_p], xs[idx_p], [0])
dk_p = -k_p * d
# dk_p = dkernel(xs[idx_p], xs[idx_p], 1.0)

# dgamma:
# rhs_p = -h_p * dk_p.dot(a[idx_p]) / lmbda * 2.0
# dlmbda:
rhs_p = h_p * ka[idx_p]

da_p, db = solve_rendered(k_p, h_p, rhs_p, lmbda)
da = np.zeros_like(a)
da[idx_p] = da_p

In [ ]:
np.hstack((da[:, None], dan[:, None]))

In [ ]:
np.abs(da[:, None] - dan[:, None]).sum()

In [ ]:
db, dbn